# Approach 3 — Modelling-Based Performance Analysis

## What this approach does

Rather than classifying practices into archetypes, this notebook answers a different
question: **given a practice's characteristics, what level of income should it be
generating — and is it over- or under-performing that expectation?**

We fit two models with increasing complexity:

1. **Linear Regression** — a transparent baseline that shows the additive contribution
   of each feature to income in plain £ terms.

2. **XGBoost** — a gradient-boosted decision tree ensemble that captures non-linear
   relationships and feature interactions that linear regression misses.

The residuals (actual minus predicted income) from the XGBoost model are then used to
flag **outliers** — practices that are meaningfully above or below their predicted
performance given their archetype.

## Why XGBoost?

| Property | Detail |
|----------|--------|
| Handles non-linearity | A flagship private practice may have a step-change in income that a linear model would flatten |
| Captures interactions | The combination of high surgeries + high hygienist presence may be worth more than the sum of its parts |
| Robust to outliers | Boosting reweights mispredicted examples rather than being dragged by them |
| Feature importances | Built-in Gain/Cover importance tells us which features drive predictions most |
| Well-validated | One of the best-performing algorithms on structured/tabular data |

## Why also run Linear Regression?

Linear regression provides a coefficient for every feature in £ terms that anyone can
read and reason about. Even if XGBoost is more accurate, the linear model is valuable
for communicating *which levers matter most* to a non-technical audience.

A large gap between linear and XGBoost R² signals that the relationship is genuinely
non-linear — worth investigating further.

## How outliers are identified

After fitting XGBoost, we compute the residual for each practice:

```
residual = actual_income - predicted_income
```

Residuals are then standardised (z-scored). Practices with |z| > 2 are flagged:
- **High Outlier** (z > +2): generating significantly more income than predicted by
  their characteristics — likely best-practice sites worth studying
- **Low Outlier** (z < -2): generating significantly less than predicted — candidates
  for intervention or closer investigation

> Note: NPS is constant in the synthetic dataset (47.7 for all practices) and cannot
> be used as a modelling target. `total_income_est` is used instead. With real data,
> NPS and patient satisfaction can be added as additional targets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

BLUE   = '#4C72B0'
ORANGE = '#DD8452'
GREEN  = '#55A868'
RED    = '#C44E52'
GREY   = '#8C8C8C'
RANDOM_STATE = 42

sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

NHS_VALUE_PER_UDA = 28.0

df = pd.read_csv('master.csv')
print(f'Loaded {len(df):,} practices')

---
## Feature engineering and target variable

In [ ]:
df['nhs_income_est']   = df['uda'].fillna(0) * NHS_VALUE_PER_UDA
df['private_income']   = df['privateincome'].fillna(0)
df['total_income_est'] = df['private_income'] + df['nhs_income_est']

# One-hot encode region (drop first to avoid dummy trap)
region_dummies = pd.get_dummies(df['region'], prefix='region', drop_first=True, dtype=float)
df = pd.concat([df, region_dummies], axis=1)
region_cols = region_dummies.columns.tolist()

base_features = [
    'numberofsurgeries',
    'unique_staff_ids',
    'uda',
    'position_hygienist',
    'contractualhours_dentist',
] + region_cols

X = df[base_features].fillna(0).values
y = df['total_income_est'].values

print(f'Features: {base_features}')
print(f'Target: total_income_est  |  Mean: £{y.mean():,.0f}  |  Std: £{y.std():,.0f}')

---
## Model 1 — Linear Regression (baseline)

Linear regression fits a straight-line relationship between each feature and the target.
The coefficient for each feature tells us: *holding everything else constant, what is
the marginal £ change in total income for a one-unit increase in this feature?*

This makes the model highly interpretable but it cannot capture:
- Non-linear effects (e.g. income grows faster than linearly with surgeries above 6)
- Interaction effects (e.g. hygienist presence matters more in private-heavy practices)

We report both in-sample R² and 5-fold cross-validated R² to detect any overfitting.

In [ ]:
lr = LinearRegression()
lr.fit(X, y)

y_pred_lr   = lr.predict(X)
r2_lr       = r2_score(y, y_pred_lr)
cv_r2_lr    = cross_val_score(lr, X, y, cv=5, scoring='r2').mean()

coef_df = (
    pd.DataFrame({'feature': base_features, 'coefficient_£': lr.coef_})
    .sort_values('coefficient_£', key=abs, ascending=False)
    .reset_index(drop=True)
)

print(f'Linear Regression')
print(f'  In-sample R2    : {r2_lr:.3f}')
print(f'  5-fold CV R2    : {cv_r2_lr:.3f}')
print(f'  Intercept       : £{lr.intercept_:,.0f}')
print()
display(coef_df)

# Coefficient plot
fig, ax = plt.subplots(figsize=(9, 5))
colors = [GREEN if v > 0 else RED for v in coef_df['coefficient_£']]
ax.barh(coef_df['feature'], coef_df['coefficient_£'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Coefficient (£ change per unit increase in feature)')
ax.set_title('Linear Regression — Feature Coefficients', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.show()

---
## Model 2 — XGBoost

### How XGBoost works

XGBoost builds an **ensemble of decision trees sequentially**. Each tree is trained to
correct the residual errors of all the trees before it (gradient boosting). The final
prediction is the sum of all trees.

Key hyperparameters used:

| Parameter | Value | Why |
|-----------|-------|-----|
| `n_estimators` | 300 | Number of trees — more trees = finer corrections, but diminishing returns |
| `learning_rate` | 0.05 | Step size per tree — small value forces more trees but reduces overfitting |
| `max_depth` | 4 | Maximum tree depth — limits complexity to avoid overfitting |
| `subsample` | 0.8 | Row sampling per tree — adds randomness to reduce variance |
| `colsample_bytree` | 0.8 | Feature sampling per tree — further reduces correlation between trees |

### Feature importances

XGBoost reports **Gain importance** — the average improvement in prediction accuracy
brought by a feature across all the splits that used it. Higher gain = more useful feature.

In [ ]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbosity=0,
)
xgb.fit(X, y)

y_pred_xgb = xgb.predict(X)
r2_xgb     = r2_score(y, y_pred_xgb)
cv_r2_xgb  = cross_val_score(xgb, X, y, cv=5, scoring='r2').mean()

imp_df = (
    pd.DataFrame({'feature': base_features, 'importance': xgb.feature_importances_})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

print(f'XGBoost')
print(f'  In-sample R2    : {r2_xgb:.3f}')
print(f'  5-fold CV R2    : {cv_r2_xgb:.3f}')
print(f'  R2 gain vs LR   : {cv_r2_xgb - cv_r2_lr:+.3f}  (positive = XGBoost captures additional non-linear structure)')
print()
display(imp_df)

fig, ax = plt.subplots(figsize=(9, 5))
colors = sns.color_palette('viridis_r', len(imp_df))
ax.barh(imp_df['feature'], imp_df['importance'], color=colors, edgecolor='white')
ax.set_xlabel('Feature importance (Gain)')
ax.set_title('XGBoost — Feature Importances', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## Model comparison

Plotting actual vs predicted income for both models shows where predictions diverge
from reality. Points far from the diagonal are potential outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Actual vs Predicted Income', fontsize=13, fontweight='bold')

max_val = max(y.max(), y_pred_lr.max(), y_pred_xgb.max())

for ax, y_pred, title, color, r2 in [
    (axes[0], y_pred_lr,  f'Linear Regression  (CV R2 = {cv_r2_lr:.3f})',  BLUE,   cv_r2_lr),
    (axes[1], y_pred_xgb, f'XGBoost  (CV R2 = {cv_r2_xgb:.3f})',           ORANGE, cv_r2_xgb),
]:
    ax.scatter(y_pred, y, alpha=0.35, s=25, color=color)
    ax.plot([0, max_val], [0, max_val], color='black', lw=1, ls='--', label='Perfect prediction')
    ax.set_xlabel('Predicted income (£)')
    ax.set_ylabel('Actual income (£)')
    ax.set_title(title)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Residual analysis

Residuals should be:
- **Centred around zero** — no systematic bias
- **Approximately normally distributed** — no heavy tails suggesting structural effects
- **Homoscedastic** — consistent spread across the income range (no funnel shape)

Systematic patterns in residuals (e.g. model underpredicts for large practices) indicate
missing features or non-linearities that could be addressed with additional data.

In [ ]:
residuals = y - y_pred_xgb
resid_z   = (residuals - residuals.mean()) / residuals.std()

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('XGBoost — Residual Diagnostics', fontsize=13, fontweight='bold')

# Distribution
axes[0].hist(residuals, bins=30, color=BLUE, edgecolor='white')
axes[0].axvline(0, color=RED, lw=1.5, ls='--')
axes[0].axvline(residuals.mean(), color=ORANGE, lw=1.5, ls='--',
                label=f'Mean {residuals.mean():+,.0f}')
axes[0].set_xlabel('Residual (£)')
axes[0].set_title('Residual distribution')
axes[0].legend(fontsize=8)

# Residual vs predicted
axes[1].scatter(y_pred_xgb, residuals, alpha=0.35, s=25, color=ORANGE)
axes[1].axhline(0, color=RED, lw=1.5, ls='--')
axes[1].set_xlabel('Predicted income (£)')
axes[1].set_ylabel('Residual (£)')
axes[1].set_title('Residuals vs predicted (homoscedasticity check)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))

# Z-score distribution
axes[2].hist(resid_z, bins=30, color=GREEN, edgecolor='white')
axes[2].axvline(-2, color=RED, lw=1.5, ls='--', label='|z| = 2 threshold')
axes[2].axvline( 2, color=RED, lw=1.5, ls='--')
n_high = (resid_z >  2).sum()
n_low  = (resid_z < -2).sum()
axes[2].set_xlabel('Standardised residual (z-score)')
axes[2].set_title(f'Z-scored residuals  |  High: {n_high}  Low: {n_low}')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Performance tier classification

Practices are assigned a performance tier based on their standardised residual:

| Tier | z-score | Interpretation |
|------|---------|----------------|
| High Outlier | > +2 | Materially outperforming predicted income — potential best-practice sites |
| Expected | -2 to +2 | Performing within the normal range for practices with similar characteristics |
| Low Outlier | < -2 | Materially underperforming — priority for investigation or support |

In [ ]:
tier = pd.Series('Expected', index=df.index)
tier[resid_z >  2] = 'High Outlier'
tier[resid_z < -2] = 'Low Outlier'

df['predicted_income']           = np.round(y_pred_xgb, 2)
df['income_residual']            = np.round(residuals, 2)
df['residual_z']                 = np.round(resid_z, 3)
df['predicted_performance_tier'] = tier

tier_colors = {'High Outlier': GREEN, 'Expected': GREY, 'Low Outlier': RED}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Performance Tiers', fontsize=13, fontweight='bold')

# Scatter: actual vs predicted coloured by tier
for t, color in tier_colors.items():
    sub = df[df['predicted_performance_tier'] == t]
    axes[0].scatter(sub['predicted_income'], sub['total_income_est'],
                    alpha=0.5, s=35, color=color, label=f'{t} (n={len(sub)})',
                    zorder=3 if t != 'Expected' else 1)
axes[0].plot([0, y.max()], [0, y.max()], color='black', lw=1, ls='--')
axes[0].set_xlabel('Predicted income (£)')
axes[0].set_ylabel('Actual income (£)')
axes[0].set_title('Actual vs predicted by tier')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))
axes[0].legend(fontsize=9)

# Tier count bar
tier_counts = df['predicted_performance_tier'].value_counts()
bars = axes[1].bar(tier_counts.index, tier_counts.values,
                   color=[tier_colors[t] for t in tier_counts.index], edgecolor='white')
axes[1].bar_label(bars, padding=3)
axes[1].set_title('Practice count by performance tier')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(tier.value_counts().to_string())

---
## Outlier spotlight

High Outliers are practices generating substantially more income than their size,
workforce and UDA profile would predict. These are candidates for:
- Best-practice case studies
- Identifying what is driving outperformance (pricing, case mix, efficiency)

Low Outliers generate substantially less than predicted. These warrant:
- Operational review (staffing issues, high cancellation, low utilisation)
- Commercial review (pricing below market, under-developed private mix)
- Leadership or management investigation

In [ ]:
outlier_cols = [
    'practicename', 'region', 'numberofsurgeries', 'unique_staff_ids',
    'total_income_est', 'predicted_income', 'income_residual',
    'predicted_performance_tier',
]

high_out = (
    df[df['predicted_performance_tier'] == 'High Outlier'][outlier_cols]
    .sort_values('income_residual', ascending=False)
    .reset_index(drop=True)
)
low_out = (
    df[df['predicted_performance_tier'] == 'Low Outlier'][outlier_cols]
    .sort_values('income_residual')
    .reset_index(drop=True)
)

def style_outliers(styled_df, color):
    return styled_df.format({
        'total_income_est': '£{:,.0f}',
        'predicted_income': '£{:,.0f}',
        'income_residual':  '£{:+,.0f}',
    }).set_properties(**{'background-color': color}, subset=['income_residual'])

if len(high_out):
    print(f'High Outliers ({len(high_out)} practices):')
    display(style_outliers(high_out.style, '#d4edda'))

if len(low_out):
    print(f'Low Outliers ({len(low_out)} practices):')
    display(style_outliers(low_out.style, '#f8d7da'))

---
## Performance tier by archetype

If you have already run `01_rules_based.ipynb`, we can show the distribution of
outlier tiers within each archetype cell. This reveals whether certain archetypes
systematically over- or under-perform their predicted income.

In [ ]:
import os

if os.path.exists('archetypes_rules.csv'):
    rules = pd.read_csv('archetypes_rules.csv')[['practicekey', 'archetype_size', 'archetype_model', 'archetype_rules']]
    merged = df.merge(rules, on='practicekey', how='left')

    tier_by_arch = (
        pd.crosstab(
            merged['archetype_rules'],
            merged['predicted_performance_tier'],
        )
        .reindex(columns=['High Outlier', 'Expected', 'Low Outlier'], fill_value=0)
        .sort_index()
    )

    fig, ax = plt.subplots(figsize=(11, 6))
    tier_by_arch.plot(
        kind='barh', stacked=True, ax=ax,
        color=[GREEN, GREY, RED], edgecolor='white'
    )
    ax.set_title('Performance Tier by Archetype (Rules-Based)', fontweight='bold')
    ax.set_xlabel('Number of practices')
    ax.set_ylabel('')
    ax.legend(title='Tier', loc='lower right', fontsize=9)
    plt.tight_layout()
    plt.show()

    display(tier_by_arch)
else:
    print('archetypes_rules.csv not found. Run 01_rules_based.ipynb first to enable this view.')

---
## Save output

In [ ]:
out_cols = [
    'practicekey', 'practicename', 'region',
    'numberofsurgeries', 'unique_staff_ids',
    'private_income', 'nhs_income_est', 'total_income_est',
    'predicted_income', 'income_residual', 'residual_z',
    'predicted_performance_tier',
]
df[out_cols].to_csv('archetypes_modelling.csv', index=False)
print(f'Saved archetypes_modelling.csv  ({len(df):,} rows)')
df[out_cols].head()